In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# ------------------------------------------------------------------------------
# Helper Functions (Simplified for Focus on Loss)
# ------------------------------------------------------------------------------

def generator_step(generator, discriminator, optimizer_g, loss_type, real_data, batch_size, latent_dim, device):
    """
    Performs a single generator update step.

    Args:
        generator: The generator model.
        discriminator: The discriminator model.
        optimizer_g: The optimizer for the generator.
        loss_type: 'minimax', 'lsgan', or 'wasserstein'.
        real_data: Real data tensor.
        batch_size: The batch size.
        latent_dim: Dimension of the latent space.
        device: The device to run on (CPU or GPU).

    Returns:
        Generator loss.
    """
    optimizer_g.zero_grad()
    noise = torch.randn(batch_size, latent_dim).to(device)
    fake_data = generator(noise)
    d_fake = discriminator(fake_data)

    if loss_type == 'minimax':
        g_loss = -torch.mean(torch.log(d_fake + 1e-8))  # Add small epsilon for numerical stability
    elif loss_type == 'lsgan':
        g_loss = torch.mean((d_fake - 1)**2)
    elif loss_type == 'wasserstein':
        g_loss = -torch.mean(d_fake)
    else:
        raise ValueError(f"Invalid loss_type: {loss_type}")

    g_loss.backward()
    optimizer_g.step()
    return g_loss.item()

def discriminator_step(generator, discriminator, optimizer_d, loss_type, real_data, batch_size, latent_dim, device, gradient_penalty=None):
    """
    Performs a single discriminator update step.

    Args:
        generator: The generator model.
        discriminator: The discriminator model.
        optimizer_d: The optimizer for the discriminator.
        loss_type: 'minimax', 'lsgan', or 'wasserstein'.
        real_data: Real data tensor.
        batch_size: The batch size.
        latent_dim: Dimension of the latent space.
        device: The device to run on (CPU or GPU).
        gradient_penalty: Optional gradient penalty function for Wasserstein-GP.

    Returns:
        Discriminator loss.
    """
    optimizer_d.zero_grad()

    # Real data loss
    d_real = discriminator(real_data)
    if loss_type == 'minimax':
        d_loss_real = -torch.mean(torch.log(d_real + 1e-8))
    elif loss_type == 'lsgan':
        d_loss_real = torch.mean((d_real - 1)**2)
    elif loss_type == 'wasserstein':
        d_loss_real = -torch.mean(d_real)
    else:
        raise ValueError(f"Invalid loss_type: {loss_type}")

    # Fake data loss
    noise = torch.randn(batch_size, latent_dim).to(device)
    fake_data = generator(noise).detach()  # Detach to avoid generator updates
    d_fake = discriminator(fake_data)
    if loss_type == 'minimax':
        d_loss_fake = -torch.mean(torch.log(1 - d_fake + 1e-8))
    elif loss_type == 'lsgan':
        d_loss_fake = torch.mean(d_fake**2)
    elif loss_type == 'wasserstein':
        d_loss_fake = torch.mean(d_fake)
    else:
        raise ValueError(f"Invalid loss_type: {loss_type}")

    d_loss = d_loss_real + d_loss_fake

    if loss_type == 'wasserstein' and gradient_penalty is not None:
        gp = gradient_penalty(discriminator, real_data, fake_data, device)
        d_loss += gp

    d_loss.backward()
    optimizer_d.step()
    return d_loss.item()


def gradient_penalty(discriminator, real_data, fake_data, device):
    """
    Calculates the gradient penalty for Wasserstein GAN with Gradient Penalty (WGAN-GP).
    """
    batch_size = real_data.size(0)
    alpha = torch.rand(batch_size, 1).to(device).expand_as(real_data)
    interpolated = (alpha * real_data + (1 - alpha) * fake_data).requires_grad_(True)
    d_interpolated = discriminator(interpolated)

    grad_outputs = torch.ones(batch_size, 1).to(device)
    gradients = torch.autograd.grad(
        outputs=d_interpolated,
        inputs=interpolated,
        grad_outputs=grad_outputs,
        create_graph=True,  # Needed for the double backward pass in R1 regularization
        retain_graph=True,
    )[0]
    gradients = gradients.view(batch_size, -1)
    gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    return gradient_penalty * 10  # Lambda = 10

# ------------------------------------------------------------------------------
# Main Training Loop (Illustrative)
# ------------------------------------------------------------------------------

def train_gan(loss_type='minimax', batch_size=64, epochs=100, latent_dim=100, gp_lambda = 10):
    """
    Trains a GAN with the specified loss function.  Simplified for demonstration.

    Args:
        loss_type: 'minimax', 'lsgan', or 'wasserstein'.
        batch_size: The batch size.
        epochs: Number of training epochs.
        latent_dim: Dimension of latent space.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1. Define Generator and Discriminator (Simplified)
    generator = nn.Sequential(
        nn.Linear(latent_dim, 256),
        nn.ReLU(),
        nn.Linear(256, 256),
        nn.ReLU(),
        nn.Linear(256, 1),  # Output dimension 1 for simplicity
    ).to(device)

    discriminator = nn.Sequential(
        nn.Linear(1, 256),
        nn.ReLU(),
        nn.Linear(256, 256),
        nn.ReLU(),
        nn.Linear(256, 1),
    ).to(device)
    if loss_type == 'wasserstein':
        # No sigmoid for Wasserstein
        discriminator[-1].weight.data.clamp_(-0.01, 0.01)
    else:
        discriminator[-1] = nn.Sequential(discriminator[-1],nn.Sigmoid()) # Add Sigmoid layer for Minimax and LSGAN

    # 2. Define Optimizers
    optimizer_g = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
    optimizer_d = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

    # 3. Generate some random real data for demonstration
    real_data = torch.randn(1000, 1).to(device)  # Example 1D real data

    # 4. Training Loop
    for epoch in range(epochs):
        for i in range(len(real_data) // batch_size):
            batch_start = i * batch_size
            batch_end = (i + 1) * batch_size
            real_batch = real_data[batch_start:batch_end]

            # Discriminator step
            if loss_type == 'wasserstein':
                d_loss = discriminator_step(generator, discriminator, optimizer_d, loss_type, real_batch, batch_size, latent_dim, device, gradient_penalty)
                for p in discriminator.parameters():
                    p.data.clamp_(-0.01, 0.01)  # Clip weights for Wasserstein
            else:
                d_loss = discriminator_step(generator, discriminator, optimizer_d, loss_type, real_batch, batch_size, latent_dim, device)

            # Generator step
            g_loss = generator_step(generator, discriminator, optimizer_g, loss_type, real_batch, batch_size, latent_dim, device)

        if (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{epochs}]  Loss D: {d_loss:.4f}, Loss G: {g_loss:.4f} ({loss_type})")


if __name__ == '__main__':
    # Train with each loss type
    train_gan(loss_type='minimax', epochs=50)
    train_gan(loss_type='lsgan', epochs=50)
    train_gan(loss_type='wasserstein', epochs=50, gp_lambda=10) # Example of using WGAN-GP
